# Notebook 4: LSTM-SNP with Fuzzy Output Layer

**Dataset**: Monthly Lake Erie Levels

## Description
This notebook implements a **fuzzy output layer** for the LSTM-SNP model. The internal 
LSTM-SNP structure is kept completely intact. The final Dense(1) layer is replaced with a 
fuzzy inference system that takes the RNN hidden state h(t) and produces the prediction 
through Takagi-Sugeno rules.

The fuzzy output layer aggregates h(t) into 2 summary features via mean-pooling, then applies 
4 rules with fixed Gaussian membership functions and trainable consequent parameters.

## Theory: Fuzzy Output Layer

### LSTM-SNP Cell (Unchanged)
All internal LSTM-SNP equations remain exactly as in the baseline.

### Fuzzy Output Computation
The Dense(1) output layer is replaced with fuzzy inference:

**Input**: $h(t) \in \mathbb{R}^{units}$ from the RNN

**Feature Aggregation**: Mean-pooling into 2 summary statistics:
- $s_1 = \text{mean}(h_{1:units/2})$
- $s_2 = \text{mean}(h_{units/2+1:units})$

**Membership Functions** (Fixed Gaussian):
- $\mu_{low}(s) = \exp\left(-\frac{(s-(-1))^2}{2 \cdot 0.5^2}\right)$
- $\mu_{high}(s) = \exp\left(-\frac{(s-(+1))^2}{2 \cdot 0.5^2}\right)$

**4 Takagi-Sugeno Rules** (trainable consequents):
$y_i = a_i s_1 + b_i s_2 + c_i$

**Defuzzification**: $\hat{y} = \frac{\sum_i w_i y_i}{\sum_i w_i}$

## Model Architecture & Implementation

In [1]:
# ============================================================
# ALL IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

TensorFlow version: 2.20.0
NumPy version: 2.0.2


### LSTM-SNP Cell

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FuzzyGate(nn.Module):
    """
    Per-unit Takagi-Sugeno fuzzy gate (2 rules, fixed Gaussian MFs,
    trainable consequent parameters). Replaces a hard_sigmoid gate.

    Fixed membership functions (on the gate's linear pre-activation z):
      mu_low(z)  = exp(-(z - (-1))^2 / (2*0.5^2))
      mu_high(z) = exp(-(z - (+1))^2 / (2*0.5^2))

    Rules (consequents are trainable, per hidden unit):
      IF z is low  -> y_low  = a_low  * z + b_low
      IF z is high -> y_high = a_high * z + b_high

    Output = sigmoid( (w_low*y_low + w_high*y_high) / (w_low + w_high + eps) )
    """
    def __init__(self, hidden_size, sigma=0.5):
        super().__init__()
        self.hidden_size = hidden_size
        self.sigma = sigma

        # Trainable consequent parameters (per hidden unit)
        self.a_low = nn.Parameter(torch.ones(hidden_size) * 0.5)
        self.b_low = nn.Parameter(torch.zeros(hidden_size))
        self.a_high = nn.Parameter(torch.ones(hidden_size) * 0.5)
        self.b_high = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, z):
        mu_low = torch.exp(-(z - (-1.0))**2 / (2 * self.sigma**2))
        mu_high = torch.exp(-(z - (1.0))**2 / (2 * self.sigma**2))

        y_low = self.a_low * z + self.b_low
        y_high = self.a_high * z + self.b_high

        numerator = mu_low * y_low + mu_high * y_high
        denominator = mu_low + mu_high + 1e-8

        gate_raw = numerator / denominator
        return torch.sigmoid(gate_raw)


class LSTMSNPCell(nn.Module):
    """
    LSTM-SNP Cell with Fuzzy Gate Replacement.

    Gates:
      r(t) = FuzzyGate(W_r x(t) + U_r u(t-1) + b_r)   [reset]
      c(t) = FuzzyGate(W_c x(t) + U_c u(t-1) + b_c)   [consumption]
      o(t) = FuzzyGate(W_o x(t) + U_o u(t-1) + b_o)   [output/generation]
      a(t) = tanh(W_a x(t) + U_a u(t-1) + b_a)        [generated spikes — unchanged]

    State update:
      u(t) = r(t) * u(t-1) - c(t) * a(t)
      h(t) = o(t) * a(t)
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        self.W = nn.Linear(input_size, 4 * hidden_size, bias=True)
        self.U = nn.Linear(hidden_size, 4 * hidden_size, bias=True)

        nn.init.xavier_uniform_(self.W.weight)
        nn.init.orthogonal_(self.U.weight)

        self.fuzzy_r = FuzzyGate(hidden_size)
        self.fuzzy_c = FuzzyGate(hidden_size)
        self.fuzzy_o = FuzzyGate(hidden_size)

    def forward(self, x, u_prev):
        z = self.W(x) + self.U(u_prev)

        z0 = z[:, :self.hidden_size]
        z1 = z[:, self.hidden_size:2*self.hidden_size]
        z2 = z[:, 2*self.hidden_size:3*self.hidden_size]
        z3 = z[:, 3*self.hidden_size:]

        r = self.fuzzy_r(z0)   # reset (fuzzy)
        c = self.fuzzy_c(z1)   # consumption (fuzzy)
        o = self.fuzzy_o(z2)   # output/generation (fuzzy)
        a = torch.tanh(z3)     # generated spikes (unchanged)

        u = r * u_prev - c * a
        h = o * a

        return h, u


class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = LSTMSNPCell(input_size, hidden_size)
        self.out = nn.Linear(hidden_size, 1)
        self.u = None

    def reset_states(self, batch_size, device):
        self.u = torch.zeros(batch_size, self.hidden_size, device=device)

    def detach_states(self):
        if self.u is not None:
            self.u = self.u.detach()

    def forward(self, x):
        if self.u is None or self.u.device != x.device or self.u.size(0) != x.size(0):
            self.reset_states(x.size(0), x.device)

        h, self.u = self.cell(x[:, 0, :], self.u)
        return self.out(h)


def build_model(input_dim, units):
    return RNNModel(input_size=input_dim, hidden_size=units)

### Fuzzy Output Layer

### Build Model

In [3]:
# ============================================================
# Model Construction (PyTorch) — Fuzzy Output Layer Replacement
# ============================================================
import torch
import torch.nn as nn

class FuzzyOutputLayer(nn.Module):
    """
    Fuzzy output layer: replaces nn.Linear(hidden_size, 1).
    Input: h(t) from RNN, shape [batch, units]
    Output: scalar prediction, shape [batch, 1]

    Aggregates h(t) into 2 summary features via mean-pooling:
      s1 = mean(h[:, :units//2])
      s2 = mean(h[:, units//2:])
    Applies 4 Takagi-Sugeno rules, fixed Gaussian MFs,
    trainable consequent parameters.
    """
    def __init__(self, units_in, sigma=0.5):
        super().__init__()
        self.units_in = units_in
        self.mu_low = -1.0
        self.mu_high = 1.0
        self.sigma = sigma

        self.rule_a = nn.Parameter(torch.empty(4))
        self.rule_b = nn.Parameter(torch.empty(4))
        self.rule_c = nn.Parameter(torch.zeros(4))
        nn.init.xavier_uniform_(self.rule_a.view(1, -1))
        nn.init.xavier_uniform_(self.rule_b.view(1, -1))

    def _gaussian_mf(self, x, center):
        return torch.exp(-(x - center)**2 / (2.0 * self.sigma**2))

    def forward(self, h):
        half = self.units_in // 2
        s1 = h[:, :half].mean(dim=-1, keepdim=True)
        s2 = h[:, half:].mean(dim=-1, keepdim=True)

        mu_low_s1 = self._gaussian_mf(s1, self.mu_low)
        mu_high_s1 = self._gaussian_mf(s1, self.mu_high)
        mu_low_s2 = self._gaussian_mf(s2, self.mu_low)
        mu_high_s2 = self._gaussian_mf(s2, self.mu_high)

        w1 = mu_low_s1 * mu_low_s2
        w2 = mu_low_s1 * mu_high_s2
        w3 = mu_high_s1 * mu_low_s2
        w4 = mu_high_s1 * mu_high_s2

        y1 = self.rule_a[0] * s1 + self.rule_b[0] * s2 + self.rule_c[0]
        y2 = self.rule_a[1] * s1 + self.rule_b[1] * s2 + self.rule_c[1]
        y3 = self.rule_a[2] * s1 + self.rule_b[2] * s2 + self.rule_c[2]
        y4 = self.rule_a[3] * s1 + self.rule_b[3] * s2 + self.rule_c[3]

        numerator = w1 * y1 + w2 * y2 + w3 * y3 + w4 * y4
        denominator = w1 + w2 + w3 + w4 + 1e-8

        return numerator / denominator


class LSTMSNPCell(nn.Module):
    """Original LSTM-SNP cell — unchanged (hard_sigmoid gates)."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        self.W = nn.Linear(input_size, 4 * hidden_size, bias=True)
        self.U = nn.Linear(hidden_size, 4 * hidden_size, bias=True)

        nn.init.xavier_uniform_(self.W.weight)
        nn.init.orthogonal_(self.U.weight)

    def forward(self, x, u_prev):
        z = self.W(x) + self.U(u_prev)

        z0 = z[:, :self.hidden_size]
        z1 = z[:, self.hidden_size:2*self.hidden_size]
        z2 = z[:, 2*self.hidden_size:3*self.hidden_size]
        z3 = z[:, 3*self.hidden_size:]

        r = torch.nn.functional.hardsigmoid(z0)
        c = torch.nn.functional.hardsigmoid(z1)
        o = torch.nn.functional.hardsigmoid(z2)
        a = torch.tanh(z3)

        u = r * u_prev - c * a
        h = o * a

        return h, u


class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = LSTMSNPCell(input_size, hidden_size)
        self.out = FuzzyOutputLayer(hidden_size)
        self.u = None

    def reset_states(self, batch_size, device):
        self.u = torch.zeros(batch_size, self.hidden_size, device=device)

    def detach_states(self):
        if self.u is not None:
            self.u = self.u.detach()

    def forward(self, x):
        if self.u is None or self.u.device != x.device or self.u.size(0) != x.size(0):
            self.reset_states(x.size(0), x.device)

        h, self.u = self.cell(x[:, 0, :], self.u)
        return self.out(h)


def build_model(input_dim, units):
    return RNNModel(input_size=input_dim, hidden_size=units)

In [4]:
# Quick model check
model = build_model(input_dim=1, units=8)
print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters())}")

RNNModel(
  (cell): LSTMSNPCell(
    (W): Linear(in_features=1, out_features=32, bias=True)
    (U): Linear(in_features=8, out_features=32, bias=True)
  )
  (out): FuzzyOutputLayer()
)
Total params: 364


## Data Pipeline — Monthly Lake Erie Levels

In [5]:
import os
for f in os.listdir('/kaggle/input/datasets/satabartosarkar123/monthly-lake-erie-levels-1921-19-csv'):
    print(f)

monthly-lake-erie-levels-1921-19.csv


In [6]:
# ============================================================
# 1. Load Time Series Data
# ============================================================
series = pd.read_csv(
    '/kaggle/input/datasets/satabartosarkar123/monthly-lake-erie-levels-1921-19-csv/monthly-lake-erie-levels-1921-19.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)
raw_values = series.values.flatten()
print(f"Data shape: {raw_values.shape}")
print(f"First 5 values: {raw_values[:5]}")

Data shape: (600,)
First 5 values: [14.763 14.649 15.085 16.376 16.926]


In [7]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

diff_values = difference(raw_values, 1)

In [8]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=1)
# ============================================================

def timeseries_to_supervised(data, lag=1):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

supervised = timeseries_to_supervised(diff_values, 1)
print(f"Supervised data shape: {supervised.shape}")

Supervised data shape: (599, 2)


In [9]:
#Train-validation-test split chronological 

import numpy as np
from sklearn.preprocessing import MinMaxScaler

# ============================================================
# 4. Chronological Percentage-Based Split (80 / 10 / 10)
# ============================================================
n_samples = len(supervised)

train_end = int(n_samples * 0.80)
val_end = int(n_samples * 0.90)

train = supervised[:train_end]
val   = supervised[train_end:val_end]
test  = supervised[val_end:]

print(f"Train set shape:      {train.shape}")
print(f"Validation set shape: {val.shape}")
print(f"Test set shape:       {test.shape}")

# ============================================================
# 5. Feature Scaling (Fit on Train ONLY)
# ============================================================
scaler = MinMaxScaler(feature_range=(-1, 1))

# Fit scaler strictly on training data
scaler.fit(train)

# Transform all splits using the fitted training scaler
train_scaled = scaler.transform(train)
val_scaled   = scaler.transform(val)
test_scaled  = scaler.transform(test)

print("\nScaling complete.")
print(f"Train Scaled Range: ({train_scaled.min():.2f}, {train_scaled.max():.2f})")
print(f"Val Scaled Range:   ({val_scaled.min():.2f}, {val_scaled.max():.2f})")
print(f"Test Scaled Range:  ({test_scaled.min():.2f}, {test_scaled.max():.2f})")

Train set shape:      (479, 2)
Validation set shape: (60, 2)
Test set shape:       (60, 2)

Scaling complete.
Train Scaled Range: (-1.00, 1.00)
Val Scaled Range:   (-0.92, 0.19)
Test Scaled Range:  (-1.00, 0.11)


In [10]:
# ============================================================
# 6. Reshape for RNN Input
# ============================================================

X_train, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))

X_test, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (479, 1, 1), y_train shape: (479,)


## Training Loop

In [ ]:
# ============================================================
# 60-Run Experiment Protocol (PyTorch)
# ============================================================
all_rmse = []
all_mse = []
all_nmse = []
all_predictions = []
all_losses = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n--- [PyTorch] RUNNING ON {device} ---\n")
for run in range(60):
    print(f'\n===== RUN {run+1}/60 =====')
    np.random.seed(run)
    torch.manual_seed(run)
    model = build_model(input_dim=1, units=8).to(device)
    
    # Initialize consumption gate bias to 1.0 (forget gate equivalent)
    if hasattr(model.cell, 'U') and True:
        with torch.no_grad():
            model.cell.U.bias.data[model.hidden_size:2*model.hidden_size] = 1.0
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    run_losses = []
    
    # Pre-tensorize training data
    if False:
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    else:
        # X_train is (batch, 1, input_dim)
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    n_samples = X_train_t.size(0)
    for epoch in range(100):
        model.train()
        model.reset_states(1, device) # batch_size=1
        
        epoch_loss = 0.0
        
        for i in range(n_samples):
            x_i = X_train_t[i:i+1] # (1, 1, input_dim)
            y_i = y_train_t[i:i+1] # (1,)
            
            optimizer.zero_grad()
            
            # Forward pass
            pred = model(x_i)
            loss = criterion(pred.squeeze(-1), y_i)
            
            # Backward and optimize
            loss.backward()
            
            # Gradient clipping for FuzzyGate variants
            if 4 in [3, 5]:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
            optimizer.step()
            
            # Detach hidden state so BPTT doesn't go all the way back to t=0
            model.detach_states()
            
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / n_samples
        run_losses.append(avg_loss)
        print(f"Epoch {epoch+1}/100 completed. Loss: {avg_loss:.6f}")
    all_losses.append(run_losses)
    print(f'Training complete for run {run+1}')
    # Warm-up: condition hidden states on training data
    model.eval()
    with torch.no_grad():
        for i in range(len(train_scaled)):
            X_raw = train_scaled[i, 0:-1]
            X_input = torch.tensor(X_raw, dtype=torch.float32).view(1, 1, len(X_raw)).to(device)
            model(X_input)
    # Test predictions (single-step)
    predictions = []
    model.eval()
    with torch.no_grad():
        for i in range(len(test_scaled)):
            X, y = test_scaled[i, 0:-1], test_scaled[i, -1]
            X_input = torch.tensor(X, dtype=torch.float32).view(1, 1, len(X)).to(device)
            yhat = model(X_input).item()
            # Invert scaling
            new_row = [x for x in X] + [yhat]
            array = np.array(new_row).reshape(1, len(new_row))
            inverted = scaler.inverse_transform(array)[0, -1]
            # Invert differencing
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)
            expected = raw_values[len(train) + i + 1]
            print(f'Month={i+1}, Predicted={inverted:.4f}, Expected={expected:.4f}')
    # Compute metrics
    actual = raw_values[-len(predictions):]
    rmse = sqrt(mean_squared_error(actual, predictions))
    mse = mean_squared_error(actual, predictions)
    meanV = np.mean(actual)
    dominator = np.linalg.norm(np.array(predictions) - meanV, 2)
    nmse = mse / np.power(dominator, 2)
    all_rmse.append(rmse)
    all_mse.append(mse)
    all_nmse.append(nmse)
    all_predictions.append(predictions)
    print(f'Run {run+1} — RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')


--- [PyTorch] RUNNING ON cuda ---


===== RUN 1/60 =====
Epoch 1/100 completed. Loss: 0.173790
Epoch 2/100 completed. Loss: 0.097791
Epoch 3/100 completed. Loss: 0.091918
Epoch 4/100 completed. Loss: 0.081095
Epoch 5/100 completed. Loss: 0.069574
Epoch 6/100 completed. Loss: 0.065265
Epoch 7/100 completed. Loss: 0.064121
Epoch 8/100 completed. Loss: 0.063772
Epoch 9/100 completed. Loss: 0.063584
Epoch 10/100 completed. Loss: 0.063426
Epoch 11/100 completed. Loss: 0.063272
Epoch 12/100 completed. Loss: 0.063117
Epoch 13/100 completed. Loss: 0.062962
Epoch 14/100 completed. Loss: 0.062805
Epoch 15/100 completed. Loss: 0.062646
Epoch 16/100 completed. Loss: 0.062485
Epoch 17/100 completed. Loss: 0.062322
Epoch 18/100 completed. Loss: 0.062154
Epoch 19/100 completed. Loss: 0.061980
Epoch 20/100 completed. Loss: 0.061801
Epoch 21/100 completed. Loss: 0.061613
Epoch 22/100 completed. Loss: 0.061417
Epoch 23/100 completed. Loss: 0.061210
Epoch 24/100 completed. Loss: 0.060992
Epoch 25/100 co

## Results

In [ ]:
# ============================================================
# Summary Statistics (60 runs)
# ============================================================
print('\n===== FINAL RESULTS (60 runs) =====')
print(f'RMSE: {np.mean(all_rmse):.6f} ± {np.std(all_rmse):.6f}')
print(f'MSE:  {np.mean(all_mse):.6f} ± {np.std(all_mse):.6f}')
print(f'NMSE: {np.mean(all_nmse):.10f} ± {np.std(all_nmse):.10f}')
best_idx = np.argmin(all_rmse)
print(f'\nBest run: {best_idx+1}')
print(f'  RMSE: {all_rmse[best_idx]:.6f}')
print(f'  MSE:  {all_mse[best_idx]:.6f}')
print(f'  NMSE: {all_nmse[best_idx]:.10f}')

In [ ]:
# ============================================================
# Predictions vs Actual (Best Run)
# ============================================================
best_predictions = all_predictions[best_idx]
actual = raw_values[-len(best_predictions):]

plt.figure(figsize=(12, 5))
plt.plot(actual, label='Actual', color='blue', linewidth=1.5)
plt.plot(best_predictions, label='Predicted (Best Run)', color='red',
         linewidth=1.5, linestyle='--')
plt.title('Fuzzy Output Layer — Dow Jones Industrial Index\nPredictions vs Actual (Best of 60 runs)')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# Loss Curve (Best Run)
# ============================================================
plt.figure(figsize=(12, 4))
plt.plot(all_losses[best_idx], color='green', linewidth=1.0)
plt.title('Fuzzy Output Layer — Dow Jones Industrial Index\nTraining Loss (Best Run)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# Final Metrics Summary
# ============================================================
print('=== Best Run Metrics ===')
print(f'RMSE: {all_rmse[best_idx]:.6f}')
print(f'MSE:  {all_mse[best_idx]:.6f}')
print(f'NMSE: {all_nmse[best_idx]:.10f}')

In [ ]:
# ============================================================
# Predictions vs Actual (Best Run)
# ============================================================

actual = raw_values[-60:]
best_predictions = all_predictions[best_idx]

plt.figure(figsize=(12, 5))
plt.plot(actual, label='Actual', color='blue', linewidth=1.5)
plt.plot(best_predictions, label='Predicted (Best Run)', color='red',
         linewidth=1.5, linestyle='--')
plt.title('Fuzzy Output Layer — Monthly Lake Erie Levels\nPredictions vs Actual (Best of 30 runs)')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# Loss Curve (Best Run)
# ============================================================

plt.figure(figsize=(12, 4))
plt.plot(all_losses[best_idx], color='green', linewidth=1.0)
plt.title('Fuzzy Output Layer — Monthly Lake Erie Levels\nTraining Loss (Best Run)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# Final Metrics Summary
# ============================================================

print('=== Best Run Metrics ===')
print(f'RMSE: {all_rmse[best_idx]:.6f}')
print(f'MSE:  {all_mse[best_idx]:.6f}')
print(f'NMSE: {all_nmse[best_idx]:.10f}')

## Observations

### Fuzzy Output Layer on Monthly Lake Erie Levels

**Run the notebook to generate results and fill in observations:**

1. **Prediction Quality**: Compare RMSE/MSE/NMSE with other variants
2. **Training Stability**: Examine loss curves for convergence behavior
3. **Prediction Tracking**: Assess how well predictions track actual values
4. **Computational Cost**: Note training time per run

*After running all 5 variant notebooks, perform cross-variant comparison to evaluate 
whether fuzzy logic improves nonlinearity handling, interpretability, and prediction performance.*